# DAVE Documents API — Users & Permissions Routes

**Prerequisite:** run `00_auth_setup.ipynb` first. Most calls here require the
**admin** role (or `USE_AUTH=false`).

| Method | Path | Description | Role |
|--------|------|-------------|------|
| GET | `/api/users` | List all users (Keycloak) | admin |
| POST | `/api/users` | Create a new user | admin |
| PUT | `/api/users/{id}` | Update a user | admin |
| DELETE | `/api/users/{id}` | Delete a user | admin |
| GET | `/api/permissions` | Get global permissions | authenticated |
| PUT | `/api/permissions` | Update global permissions | admin |

In [ ]:
import sys, os, json, requests

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from auth_state import API_BASE, auth_headers

print(f"API base: {API_BASE}")

---
## Users
### GET /api/users — List all users

In [ ]:
resp = requests.get(f"{API_BASE}/users", headers=auth_headers())
if resp.status_code == 403:
    print("403 Forbidden — admin role required.")
else:
    resp.raise_for_status()
    users = resp.json()
    print(f"{len(users)} user(s) found")
    for u in users:
        print(f"  id={u.get('id')}  email={u.get('email')}  roles={u.get('roles')}")

    sample_user_id = users[0]["id"] if users else None

### POST /api/users — Create a new user

In [ ]:
new_user_payload = {
    "email":     "testnotebook@example.com",
    "password":  "Passw0rd!",
    "firstName": "Test",
    "lastName":  "Notebook",
    "role":      "viewer",   # admin | editor | viewer
}

resp = requests.post(
    f"{API_BASE}/users",
    json=new_user_payload,
    headers=auth_headers(),
)
if resp.status_code == 403:
    print("403 Forbidden — admin role required.")
    created_user = None
elif resp.status_code == 409:
    print("User already exists.")
    created_user = None
else:
    resp.raise_for_status()
    created_user = resp.json()
    print("Created:", json.dumps(created_user, indent=2))

### PUT /api/users/{id} — Update a user

In [ ]:
user_id = (created_user or {}).get("id") or globals().get("sample_user_id")

if not user_id:
    print("No user ID available — skipping.")
else:
    resp = requests.put(
        f"{API_BASE}/users/{user_id}",
        json={
            "firstName": "TestUpdated",
            "role":      "editor",
        },
        headers=auth_headers(),
    )
    if resp.status_code == 403:
        print("403 Forbidden — admin role required.")
    else:
        resp.raise_for_status()
        print(json.dumps(resp.json(), indent=2))

### DELETE /api/users/{id} — Delete a user

In [ ]:
delete_user = False  # ← set True to delete the user created above
del_user_id = (created_user or {}).get("id")

if delete_user and del_user_id:
    resp = requests.delete(
        f"{API_BASE}/users/{del_user_id}",
        headers=auth_headers(),
    )
    if resp.status_code == 403:
        print("403 Forbidden — admin role required.")
    else:
        resp.raise_for_status()
        print(resp.json())
else:
    print("Skipped (set delete_user=True to run).")

---
## Permissions

The single global permissions document controls what roles can do across
collections, documents, chat, and settings.

### GET /api/permissions — Read global permissions

In [ ]:
resp = requests.get(f"{API_BASE}/permissions", headers=auth_headers())
if resp.status_code == 404:
    print("No permissions document found (server default is used).")
    current_permissions = None
else:
    resp.raise_for_status()
    current_permissions = resp.json()
    print(json.dumps(current_permissions, indent=2, default=str))

### PUT /api/permissions — Update global permissions

Replaces the `collections`, `document`, `chat`, and `settings` sections.
Admin role required.

In [ ]:
# Example permissions object — adjust to your needs.
# Supply only the sections you want to update; omit others to keep them.
new_permissions = {
    "collections": {
        "view":   ["admin", "editor", "viewer"],
        "create": ["admin", "editor"],
        "delete": ["admin"],
    },
    "document": {
        "view":   ["admin", "editor", "viewer"],
        "update": ["admin", "editor"],
        "delete": ["admin"],
    },
    "chat": {
        "use": ["admin", "editor", "viewer"],
    },
    "settings": {
        "pipeline": ["admin", "editor"],
    },
}

resp = requests.put(
    f"{API_BASE}/permissions",
    json=new_permissions,
    headers=auth_headers(),
)
if resp.status_code == 403:
    print("403 Forbidden — admin role required.")
else:
    resp.raise_for_status()
    print("Updated permissions:")
    print(json.dumps(resp.json(), indent=2, default=str))

### Restore original permissions (optional)

In [ ]:
restore_permissions = False  # ← set True to restore previously read permissions

if restore_permissions and current_permissions:
    restore_payload = {
        k: current_permissions[k]
        for k in ("collections", "document", "chat", "settings")
        if k in current_permissions
    }
    resp = requests.put(
        f"{API_BASE}/permissions",
        json=restore_payload,
        headers=auth_headers(),
    )
    resp.raise_for_status()
    print("Permissions restored.")
else:
    print("Skipped (set restore_permissions=True to run).")